# Testing GPU

## Import TensorFlow and other libraries

In [80]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [81]:
def normalize(input_image, real_image):
  input_image = (input_image / 127.5) - 1
  real_image = (real_image / 127.5) - 1
  return input_image, real_image

In [82]:
from matplotlib import pyplot as plt
def plot_sample_sequence(sample_sequence):
    fig = plt.figure(figsize=(20, 4))
    for i, im in enumerate(sample_sequence):
        ax = fig.add_subplot(2,10,i+1)
        ax.imshow(im, cmap='gray') 
        ax.axis('off')
    plt.show()
    plt.close()

def sequence_generator(sample_sequence):
    for i in range(sample_sequence.shape[0]):
        yield sample_sequence[i, ...]

def plot_sequence(sample_sequence):
    plot_sample_sequence(sequence_generator(sample_sequence))

def inverse_sequence_generator(sample_sequence):
    for i in range(sample_sequence.shape[-1]):
        yield sample_sequence[..., i]

def plot_inverse_sequence(sample_sequence):
    plot_sample_sequence(inverse_sequence_generator(sample_sequence))

def plot_sequence_from_tensor(sample_sequence):
    if len(sample_sequence.shape) == 4 and sample_sequence.shape[0] == 1:
        plot_sample_sequence(inverse_sequence_generator(sample_sequence[0]))
    else:
        raise ValueError("The tensor must be of shape (1, height, width, channels)")

In [83]:
import tensorflow as tf
import numpy as np

def ratio_std(previous, current):
  previous, current = normalize(previous, current)
  ratio = tf.math.divide_no_nan(current, previous)
  # ratio = current-previous
  ratio = tf.reshape(ratio, [ratio.shape[0], ratio.shape[1]*ratio.shape[2]])
  return tf.math.reduce_std(ratio, axis=1)


def ratio_loss(y_true, y_pred):

  std_pred = []
  std_true = []
  for i in range(1, y_true.shape[-1]):
    std_pred.append(ratio_std(y_pred[...,i-1], y_pred[...,i]))
    std_true.append(ratio_std(y_true[...,i-1], y_true[...,i]))
  std_pred = tf.convert_to_tensor(std_pred, dtype=tf.float32)
  std_true = tf.convert_to_tensor(std_true, dtype=tf.float32)
  print("std_true")
  print(std_true)
  print("std_pred")
  print(std_pred)
  return tf.math.reduce_sum(tf.abs(std_true - std_pred))

In [84]:
a = np.array([
        [
            [
                [1., 1., 1.],
                [1., 1., 1.],
                [1., 1., 1.],
            ],
            [
                [4., 4., 4.],
                [4., 4., 4.],
                [4., 4., 4.],
            ],
            [
                [4., 4., 4.],
                [4., 4., 4.],
                [4., 4., 4.],
            ],
            [
                [2., 2., 2.],
                [2., 2., 2.],
                [2., 2., 2.],
            ],
        ],
        [
            [
                [1., 1., 1.],
                [1., 1., 1.],
                [1., 1., 1.],
            ],
            [
                [4., 4., 4.],
                [4., 4., 4.],
                [4., 4., 4.],
            ],
            [
                [4., 4., 4.],
                [4., 4., 4.],
                [4., 4., 4.],
            ],
            [
                [2., 2., 2.],
                [2., 2., 2.],
                [2., 2., 2.],
            ],
        ]
    ])

b = np.array([
        [
            [
                [1., 1., 1.],
                [1., 1., 1.],
                [1., 1., 1.],
            ],
            [
                [2., 2., 2.],
                [2., 2., 2.],
                [2., 2., 2.],
            ],
            [
                [4., 4., 4.],
                [4., 4., 4.],
                [4., 4., 4.],
            ],
            [
                [3., 3., 3.],
                [3., 3., 3.],
                [3., 3., 3.],
            ],
        ],
        [
            [
                [1., 1., 1.],
                [1., 1., 1.],
                [1., 1., 1.],
            ],
            [
                [3., 3., 3.],
                [3., 3., 3.],
                [3., 3., 3.],
            ],
            [
                [4., 4., 4.],
                [4., 4., 4.],
                [4., 4., 4.],
            ],
            [
                [3., 3., 3.],
                [3., 3., 3.],
                [3., 3., 3.],
            ],
        ]
    ])
a = np.moveaxis(a, 1, -1)
b = np.moveaxis(b, 1, -1)
ratio_loss(a, b)

std_true
tf.Tensor(
[[0. 0.]
 [0. 0.]
 [0. 0.]], shape=(3, 2), dtype=float32)
std_pred
tf.Tensor(
[[1.110223e-16 1.110223e-16]
 [1.110223e-16 1.110223e-16]
 [0.000000e+00 0.000000e+00]], shape=(3, 2), dtype=float32)


<tf.Tensor: shape=(), dtype=float32, numpy=4.440892e-16>

In [85]:
from pathlib import Path

PATH = Path('/home/rafa/Sync/data/mnist_test_seq.npy')
sequences = np.load(PATH)
a = sequences[:,0,...]
b = sequences[:,1,...]

#convert to float32
a = a.astype(np.float32)
b = b.astype(np.float32)

# increase the number of channels
a = np.expand_dims(a, axis=-1)
b = np.expand_dims(b, axis=-1)

# move the channel axis to the end
a = np.moveaxis(a, -1, 0)
b = np.moveaxis(b, -1, 0)
a = np.moveaxis(a, 1, -1)
b = np.moveaxis(b, 1, -1)



print("a, a")
print(ratio_loss(a, a))
print("b, b")
# plot_sequence_from_tensor(b)
print(ratio_loss(b, b))
print("a, a+1")
print(ratio_loss(a, a+1))
# plot_sequence_from_tensor(a)
# plot_sequence_from_tensor(a+1)
print("a+1, a+1")
print(ratio_loss(a+1, a+1))
print("a+1, a")
print(ratio_loss(a+1, a))
print("b+1, b")
print(ratio_loss(b+1, b))
print("b+1, b+1")
print(ratio_loss(b+1, b+1))
print("a*b, a*b")
print(ratio_loss(a*b, a*b))
print("a, b")
print(ratio_loss(a, b))
print("b, a")
print(ratio_loss(b, a))





a, a
std_true
tf.Tensor(
[[1.6529831]
 [1.8129287]
 [1.6471288]
 [1.8204125]
 [1.6516242]
 [1.6876926]
 [1.6429073]
 [1.7749528]
 [1.881253 ]
 [1.7370661]
 [1.7750003]
 [1.719768 ]
 [1.6436064]
 [1.8818064]
 [1.766292 ]
 [1.7251682]
 [1.8702482]
 [1.8789673]
 [2.0098777]], shape=(19, 1), dtype=float32)
std_pred
tf.Tensor(
[[1.6529831]
 [1.8129287]
 [1.6471288]
 [1.8204125]
 [1.6516242]
 [1.6876926]
 [1.6429073]
 [1.7749528]
 [1.881253 ]
 [1.7370661]
 [1.7750003]
 [1.719768 ]
 [1.6436064]
 [1.8818064]
 [1.766292 ]
 [1.7251682]
 [1.8702482]
 [1.8789673]
 [2.0098777]], shape=(19, 1), dtype=float32)
tf.Tensor(0.0, shape=(), dtype=float32)
b, b
std_true
tf.Tensor(
[[3.60098  ]
 [4.484843 ]
 [4.4276185]
 [3.5228286]
 [4.5196137]
 [3.1858714]
 [3.8184602]
 [4.4868307]
 [4.5885286]
 [4.5396595]
 [2.4364789]
 [4.561633 ]
 [2.0319967]
 [4.5665007]
 [4.696216 ]
 [4.615556 ]
 [4.5281196]
 [4.69595  ]
 [3.9125168]], shape=(19, 1), dtype=float32)
std_pred
tf.Tensor(
[[3.60098  ]
 [4.484843 ]
 [4.427